# Context-Aware Trace Debugging with Falcon AI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/falcon-ai-page/falcon-ai/context-aware-debugging.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/falcon-ai-page/falcon-ai/context-aware-debugging.ipynb)

| Time | Difficulty |
|------|------------|
| 10 min | Beginner |

Open Falcon AI on a failing trace and run three turns: ask what went wrong, drill in with `/analyze-trace-errors`, and get a paste-ready prompt diff from `/fix-with-falcon`. You walk away with a verified prompt fix in minutes, without ever copy-pasting a trace ID or switching tabs.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- A traced project on the platform with at least one failing trace. If you don't have one, instrument any agent with the `Add tracing` step below and let it run a query that exposes a failure.
- Python 3.10+
- OpenAI API key (`OPENAI_API_KEY`)


## Install


In [ ]:
%pip install fi-instrumentation-otel traceai-openai openai

In [ ]:
import os
os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["OPENAI_API_KEY"] = "your-openai-key"

## Step 1: Add tracing to your agent

Three lines send every LLM call and tool invocation to FutureAGI as structured spans. `OpenAIInstrumentor` auto-instruments the OpenAI SDK; wrap your agent's entry point with `@tracer.agent` so each request becomes one parent span.


In [ ]:
from fi_instrumentation import register, FITracer
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="research-assistant-demo",
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
tracer = FITracer(trace_provider.get_tracer("research-assistant-demo"))

In [ ]:
from openai import OpenAI

client = OpenAI()


# Replace this with your own agent's entry point.
# The @tracer.agent decorator makes each call show up as one parent span
# in your FutureAGI Tracing project, with the OpenAI calls nested underneath.
@tracer.agent(name="my_agent")
def my_agent(user_message: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a research assistant. Provide citations to support your claims."},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content


# Asking for citations on a topic the model has no search tool for is a
# common failure mode (the model fabricates papers from training data).
# This gives Falcon AI a failing trace to analyze in the next step.
print(my_agent("What\'s the seminal paper on transformers?"))
print(my_agent("What are the key papers on contrastive learning for self-supervised vision?"))

trace_provider.force_flush()

For broader instrumentation patterns (custom spans, metadata tagging, prompt template tracking), see [Manual Tracing](https://docs.futureagi.com/docs/cookbook/quickstart/manual-tracing).

## Step 2: Turn 1: open Falcon AI on the trace and ask what went wrong

In **Tracing**, click into the failing trace so the trace detail page is the active view. Open the Falcon AI sidebar; it opens with a context chip showing the current trace ID, so every question you ask is answered against that specific trace. Type:

> What went wrong with this trace?

> **Tip.** `Cmd+K` (Mac) or `Ctrl+K` (Windows) opens Falcon AI from anywhere in the dashboard, with the current page auto-attached as a context chip.

Falcon AI reads the trace and gives an exploratory diagnosis: empty tool result, fallback to parametric memory, hallucinated paper descriptions.

![Falcon AI sidebar opened on the failing trace, with the trace context chip in the chat input and an exploratory diagnosis of the empty search result](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/falcon-ai/context-aware-debugging/turn-1-open-question.png)


## Step 3: Turn 2: drill in with /analyze-trace-errors

> /analyze-trace-errors

Two findings, both High impact: a tool dispatch issue and Hallucinated Content. Plus a quality scorecard and three recommended fixes.

![Falcon AI showing the structured /analyze-trace-errors output with category findings, severity, and a quality scorecard for the same trace](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/falcon-ai/context-aware-debugging/turn-2-analyze-trace-errors.png)

This is diagnosis with suggestions. The next turn turns the suggestion into a paste-ready diff.


## Step 4: Turn 3: get the fix with /fix-with-falcon

> /fix-with-falcon

Falcon AI returns a Current vs Replace with diff: keep the original prompt, append an empty-results instruction.

![Falcon AI fix-with-falcon output for the same trace showing What happened, Root cause in the agent, and a Current vs Replace with prompt diff](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/falcon-ai/context-aware-debugging/turn-3-fix-with-falcon.png)

The Current block is pulled directly from the LLM span, not guessed. Paste the Replace with block as your new system prompt, re-run the same query, and open the new trace: empty tool result followed by the refusal, no fabricated content.


> **Check.** You went from a failing trace to a verified prompt fix in three Falcon AI turns. No trace IDs copied, no spans expanded by hand.

## Explore further

- **[End-to-End with Falcon AI](/docs/cookbook/falcon-ai/end-to-end)**: The full lifecycle: trace, debug, evaluate, dataset, fix in one workflow
- **[Building Evaluation Datasets from Production Traces](/docs/cookbook/falcon-ai/eval-datasets-from-traces)**: Once you've fixed one trace, lock the failure pattern in as a regression set
- **[Error Feed](/docs/error-feed)**: Per-trace quality scoring and error-category drilldown